# 3. Evaluate behavior parameters



In this notebook, we will use the trained model to evaluate the dynamics behvior parameters: g, v, D for each cell. The combination of these three parameter shapes the dynamic behavior of a cell. We can archive taht by reading the experimental config file and easily:
- load the trained model
- load the data 
- evaluate the parameters
- perform short-term simulation 

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
if sys.platform.startswith("darwin"):
    os.environ['KMP_DUPLICATE_LIB_OK']='True'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
sc.settings.set_figure_params(frameon=False, dpi=30)

import pseudodynamics as pdp
os.chdir(pdp.main_dir)
print("workding directory changed to:", pdp.main_dir)

## Experimental Config

In [ ]:
tompos_config_path = "logs/tom_pos-DM_scaled_n[0, 1, 2, 3, 4, 6, 8]/pde_params_tsense/V4_config.json"
tompos_config = pdp.ExperimentConfig(config=tompos_config_path)

In [ ]:
tompos_config.find_lastest_ckpt()

In [ ]:
from pseudodynamics import models
pde_model = models.pde_params.load_from_checkpoint(
                    checkpoint_path = tompos_config.find_lastest_ckpt(), 
                    map_location='cpu')

Check whether it learns time-dependent parameters

In [ ]:
pde_model.time_sensitive

## resume dataloder

We can retrieve which single cell dataset to use from basic experimental config

In [ ]:
# load config
data_name = tompos_config.experiment_config['dataset']
print(data_name)

# load adata
adata = sc.read_h5ad(f"data/{data_name}.h5ad")
adata

In [ ]:
adata.obs['timepoint_tx_days'].unique()

We can resume a DataSet object with the single-cell data and the settings. By default we use the TwoTiempoint_AnnDS which trains the model for connecting two consecutive time points.

In [ ]:
# # To resume the original dataloader without masked-out time point
from pseudodynamics import reader
DS_sub = pdp.reader.TwoTimpepoint_AnnDS(AnnData=adata, split = 'train',
                                **tompos_config.dataset_config
                              )

In [ ]:
# To resume the full dataset with all time points
ds_config = tompos_config.dataset_config.copy()
ds_config['timepoint_idx'] = None

DS_full = reader.TwoTimpepoint_AnnDS(AnnData=adata, split = None,
                            **ds_config
                        )                       

## Predict dynamic parameters for single cell 

***pseudodynamics+*** follows a advected-diffusion equotion to model the density changes :

$$ \frac{\partial }{{\partial t}}u\left( { \mathbf{s},t} \right) = 
\underbrace{g\left( { \mathbf{s},t} \right)u\left( { \mathbf{s},t} \right)}_{\textrm{growth}}
- \underbrace{\nabla_{\mathbf{s}} \left( {v\left( { \mathbf{s},t} \right)u( \mathbf{s},t)} \right)}_{\textrm{drift}} 
+ \underbrace{\nabla_{\mathbf{s}} \left( {D\left( { \mathbf{s},t} \right)\nabla_{\mathbf{s}} u( \mathbf{s},t)} \right)}_{\textrm{diffusion}} 
$$

We takes all the captured single cell expression and its low-dimensional representation as the full cell state space. For time-dependent parameters, the growth rate ($g$), differentation rate ($v$) and Diffusion ($D$) can change by $t$ even for the same cell state $s$.

In [ ]:
# here we have a one-line code to predict the parameters for all cell and time point
g_pred_ay, v_pred_ay, D_pred_ay = pde_model.predict_param(DS_full)


print(f'the number of cell : {adata.shape[0]}')
print(f'the number of time points : {adata.uns["pop"]["t"].shape[0]}' )
print('the shape of growth rate :\t\t', g_pred_ay.shape)
print('the shape of differentaition rate :\t', v_pred_ay.shape)
print('the shape of diffusion :\t\t', D_pred_ay.shape)

In [ ]:
# we can visualize the growth rate by 
fig, ax = pdp.pl.params_in_umap(adata, prediction=g_pred_ay, param = 'growth')

Differentiation rate is a high-dimensional vector which denotes the immediate movement in every dimension of the diffusion map space. To visualize it, we calculate the vectorial sum which represent the overall distance of movement in a short time interval.

In [ ]:
v_norm = np.sqrt(np.sum(v_pred_ay**2, axis=-1))
fig, ax = pdp.pl.params_in_umap(adata, prediction=v_norm, param = r'diffe. $||v||_2$')

For computational simplicity, *pseudodynamics+* assumes an isogenic diffusion parameter, which means $D$ is the same for all dimension. Parameter $D$ also buffers the under-fit density changes during training, which makes it diffcult to interpret.

### predicting parameters for arbitrary time point 

*pseudodynamics+* learns a behavior network corresponding to each parameter. This network allows us to impute the parameters for any time point.

In [ ]:
import torch
# the cell state matrix
s = torch.from_numpy(DS_full.cellstate).float()
print(s.shape, s.dtype)

In [ ]:
# for example we impute day 5 and day 18
g_impute = []

with torch.no_grad():
    
    for t in [5,18,36]:
        t_input = torch.Tensor([t]).broadcast_to((s.shape[0],1))     # make them a tenso
        g_t = pde_model.g(s,t_input).cpu().numpy()
        g_impute.append(g_t)

    g_impute = np.stack(g_impute)
    print(g_impute.shape)

In [ ]:
fig, ax = pdp.pl.params_in_umap(adata, prediction=g_impute, param = 'growth', timepoints=[5, 18, 36], cell_of_t=False)

### continuous parameters

Next we inspect the evolution of the parameters in a time-continuous way. For the ease of visualisation, we group the parameters by cell types.

In [ ]:
# 
grouped_params = pdp.tl.continuous_params(pde_model, DS_full, 
                                        param='g',
                                        groupby_key='anno_man',  # group parameters by refined cell type, set to None if you want to keep single cell resolution
                                        n_interval=10,      # the resolution, the number of time intervals between two observed time points 
                                        device='cpu') 

grouped_param = pd.concat(grouped_params, axis=0)
grouped_param['g'] = grouped_param['g'].values 
grouped_param = grouped_param.query("`ct_of_day` > 10")   # only include cell type with more than 10 cells at the time
grouped_param['variable'] = grouped_param.variable.astype(float)

In [ ]:

import seaborn as sns

celltype_pallete = dict(zip(adata.obs['anno_man'].cat.categories, adata.uns['anno_man_colors']))

all_int_time = grouped_param['variable'].unique()
intermediate_time = all_int_time.astype(float)[np.arange(0,81,5)]
obs_T = all_int_time[np.arange(0,81,10)]

grouped_param_shorter = grouped_param.query('`variable` <= 76')

fig = plt.figure(dpi=60, figsize=(12,6), frameon=True)
fig.tight_layout(rect=[-0.5,-0.6,1.05,1])
ax = plt.gca()
g = sns.lineplot(data=grouped_param_shorter, x='variable', y='g', 
                 hue='anno_man', palette=celltype_pallete,
                 ax=ax, linewidth=2.5,  legend=False,
                 )
sns.scatterplot(data=grouped_param_shorter.query('`variable` in @obs_T'), x='variable', y='g', 
                 hue='anno_man', palette=celltype_pallete,
                 marker='o', legend=False,
                 ax=ax
                 )
g.legend(frameon=False)
sns.despine(ax=g)
g.set_ylabel("Growth rate", fontsize=12)
g.set_xlabel("Time point (Day)", fontsize=12)




In [ ]:
adata.uns

In [ ]:
adata.obs.keys()

### relative rate 

In [ ]:
ct_v = pdp.tl.agg_param(adata, param=v_norm, groupby_key='fine_anno')

# devided by HSC diff. rate at each time point
ct_v = ct_v / ct_v.loc['HSC'].values
ct_v = ct_v.iloc[1:]
ct_v

In [ ]:
fig = plt.figure(figsize=(6, 5), dpi=80)
ax = fig.gca()
sns.heatmap(data=ct_v, ax=ax)
ax.set_title('relative diffe. rate over HSC', fontsize=18)
ax.set_ylabel('cell type')
ax.set_xlabel('timepoints')

## Simulating density change

The above equation describe the change of density over time. Now with the parameters and the suggorate model, we can formulate a simulation of the density change using NeuralODE.

Here when using TwoTimpepoint_AnnDS, we take the based density from the last time point to simualate the density at the next time point.

In [ ]:
# a one line code for short term simulation( two consecutive time points) 
u_sim = pdp.tl.density_shortterm_simulation(pde_model, DataSet=DS_full, timepoints=adata.uns['pop']['t'])
u_int_all = np.concatenate([DS_full.u_b[0,None], u_sim], axis=0)

print(u_int_all.shape)

In [ ]:
# unseen time points
seen_i = tompos_config.dataset_config['timepoint_idx']
masked_i = [i for i,t in enumerate(adata.uns['pop']['t']) if i not in seen_i]

masked_t = adata.uns['pop']['t'][masked_i]
seen_t = adata.uns['pop']['t'][seen_i]

print("seen time points : ", seen_t)
print("held-out time points : ", masked_t)


The model predict pretty well in terms of the density

In [ ]:
# visualize the density at unseen time point during training and compare to ground turth
pdp.pl.params_in_umap(adata, np.log(DS_full.u_b[masked_i]+1e-5), timepoints=masked_t, param=r'$\log u$');
pdp.pl.params_in_umap(adata, np.log(u_int_all[masked_i]+1e-5), timepoints=masked_t, param=r'$\log{u_{sim}}$');

In [ ]:
# let's check the seen time points
pdp.pl.params_in_umap(adata, np.log(DS_full.u_b[seen_i]+1e-5), timepoints=seen_t, param=r'$\log u$');
pdp.pl.params_in_umap(adata, np.log(u_int_all[seen_i]+1e-5), timepoints=seen_t, param=r'$\log{u_{sim}}$');

**What's Next**  

Link single-cell dynamics behavior with molecualr evidence:
- cell cycle phase  (growth rate $g$)
- differentiation rate association test (diffe. rate $v$)